In [1]:
import pandas as pd
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem  import PorterStemmer

In [2]:
df = pd.read_csv("IMDB Dataset.csv")

In [3]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
df.shape

(50000, 2)

In [5]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [6]:
df.drop_duplicates(inplace = True)

In [7]:
df.shape

(49582, 2)

### Text Preprocessing

In [8]:
df['review'] = df['review'].str.lower()

### Tags Removing 

In [9]:
import re

In [10]:
def remove_url(text):
    text = re.sub(r"https\S+","",text)
    return text
df['review'] = df["review"].apply(remove_url)

### Punctuation Removing

In [11]:
def remove_punchuation(text):
    text = re.sub(r"[^A-Za-z0-1\s]","",text)
    return text
df['review'] = df["review"].apply(remove_punchuation)

### HTML Tags Removing

In [12]:
def remove_tags(text):
    text = re.sub(r"<.*?>","",text)
    return text
df['review'] = df["review"].apply(remove_tags)

### Removing the Stopwords

In [13]:
def remove_stopword(text):
    tokens = word_tokenize(text)
    stop_word = stopwords.words("english")
    for word in tokens:
        if word in stop_word:
            text = text.replace(word,"")
    return text

In [14]:
df['review'] = df["review"].apply(remove_stopword)

In [15]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


### Stemming

In [17]:
def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []
    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)
    return " ".join(stemmed_words)


In [18]:
df["review"] = df["review"].apply(stemming)

In [19]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,positive
1,wder ltle producti br br film techniqu unssum ...,positive
2,thought th wder wy spend tme o hot summer week...,positive
3,bsclli re fmli lttle boy jke thk re zomb close...,negative
4,petter mtte love time mey vulli stunng film wt...,positive


### Encoding

In [20]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df["sentiment"] = le.fit_transform(df["sentiment"])
y = df["sentiment"]

In [21]:
y

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 49582, dtype: int64

In [22]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,1
1,wder ltle producti br br film techniqu unssum ...,1
2,thought th wder wy spend tme o hot summer week...,1
3,bsclli re fmli lttle boy jke thk re zomb close...,0
4,petter mtte love time mey vulli stunng film wt...,1


### Vectorization

In [23]:
from sklearn.feature_extraction.text import TfidfVectorizer
tf = TfidfVectorizer(max_features = 5000)
X = tf.fit_transform(df["review"])

In [24]:
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4053702 stored elements and shape (49582, 5000)>

### Data Loading and splitting 

In [25]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size =0.2,random_state = 42)


In [26]:
from torch.utils.data import DataLoader,TensorDataset

In [27]:
X_train = X_train.toarray()
X_test = X_test.toarray()


In [28]:
import torch

In [29]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
    )
test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
    )

In [30]:
train_loader = DataLoader(train_set,shuffle = True,batch_size = 64)
test_loader = DataLoader(test_set,shuffle = True,batch_size = 64)

In [31]:
import torch.nn as nn
import torch.optim as optim

### Model Creating

In [32]:
class RNN(nn.Module):
    def __init__(self,input_size,hidden_size = 128,num_layers = 1):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        #RNN layer
        self.rnn = nn.RNN(input_size,hidden_size,num_layers,batch_first =True)

        #FC Layer
        self.fc = nn.Linear(hidden_size,1)
    def forward(self,x):
        h0 = torch.zeros(self.num_layers,x.size(0),self.hidden_size)

        out,_ = self.rnn(x,h0)

        out = self.fc(out[:,-1,:])
        return out

In [33]:
input_size = X.shape[1]
model = RNN(input_size)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

### Model Training

In [34]:
epochs  = 9
model.train()
for epoch in range(epochs):

    for xb,yb in train_loader:
        optimizer.zero_grad()
        xb = xb.unsqueeze(1)
        
        outputs = model(xb)
        outputs = torch.sigmoid(outputs.squeeze())
        
        loss = criterion(outputs,yb)
        loss.backward()
        optimizer.step()
    print(f"epoch : {epoch}, Loss : {loss.item()}")

epoch : 0, Loss : 0.2981587052345276
epoch : 1, Loss : 0.2324632853269577
epoch : 2, Loss : 0.19928577542304993
epoch : 3, Loss : 0.28340625762939453
epoch : 4, Loss : 0.19220243394374847
epoch : 5, Loss : 0.2096078097820282
epoch : 6, Loss : 0.2851909101009369
epoch : 7, Loss : 0.150479257106781
epoch : 8, Loss : 0.2480250746011734


### Model Evaluation

In [35]:
model.eval()
with torch.no_grad():
    total_correct = 0.0
    total_value = 0.0
    for xb,yb in test_loader:
        xb = xb.unsqueeze(1)
        outputs = model(xb)
        predicted = (torch.sigmoid(outputs.squeeze())>0.5).float()
        total_correct += (predicted == yb).sum().item()
        total_value += yb.size(0)
    print(f"Accuracy : {(total_correct/total_value)*100}")

Accuracy : 85.44922859735807


### Save the Model

In [36]:
import pickle
import torch

# save tfidf
with open("tfidf.pkl","wb") as f:
    pickle.dump(tf,f)

# save model
torch.save(model.state_dict(),"sentiment_model.pth")